# Jour 1 · Variables, cible et types de problèmes

> Version corrigée — réponses et raisonnement après les exercices.

## Objectifs

- identifier observation, feature et cible
- distinguer classification, régression et non supervisé
- construire X et y sans inclure d’information interdite


## Partir de la question métier

Un projet ML commence par une question dont la réponse est observable. « Anticiper les problèmes » est trop vague. « Une panne surviendra-t-elle dans les sept prochains jours ? » définit une réponse binaire et une période précise.

Une **observation** est un cas individuel : ici une machine à un instant de référence. Une **feature**, ou variable d'entrée, est une information disponible au moment de la décision. La **cible**, souvent notée `y`, est la réponse que le modèle doit apprendre à produire.

### Tableau, ligne et colonne

Dans le code, pandas stocke les données dans un **DataFrame** : un tableau avec des lignes et des colonnes. Une ligne représente une observation ; une colonne représente une variable. Cette structure ressemble à une feuille Excel, mais elle peut être filtrée et transformée par Python.



## Le rôle et le type des colonnes

![Exemple de ligne avec identifiant features et cible](../../ressources/illustrations/jour_01/01_types_colonnes.png)

Une colonne peut être **numérique** comme une température, **catégorielle** comme un état de fonctionnement, ou **binaire** lorsqu’elle ne possède que deux réponses possibles. Son format ne suffit toutefois pas à décider si elle doit être utilisée.

Un identifiant peut être écrit avec des chiffres tout en ne représentant aucune grandeur mesurable. Le donner au modèle peut l’inciter à reconnaître les machines déjà vues au lieu d’apprendre des relations générales.

Avant de construire `X` et `y`, on pose donc trois questions :

1. cette information sera-t-elle disponible au moment de prédire ?
2. décrit-elle réellement la situation observée ?
3. contient-elle directement ou indirectement la réponse future ?

Pour les questions 1 et 2, on attend **oui**. Pour la question 3, on attend **non** : une réponse **oui** révèle une fuite possible. Toute réponse ambiguë demande une vérification métier avant l'entraînement.


![Séparation des mesures en features X et cible y](../../ressources/illustrations/jour_01/01_features_cible.png)

*Les features décrivent la situation connue ; la cible représente la réponse historique à apprendre.*


## Classification, régression ou apprentissage non supervisé ?

- La **classification** prédit une catégorie : panne ou pas de panne, type d'objet, présence ou absence d'un casque.
- La **régression** prédit un nombre continu : température future, durée restante ou consommation.
- L'**apprentissage non supervisé** ne dispose pas de cible connue et cherche une structure, par exemple des groupes ou des observations atypiques.

Reconnaître un chiffre ou une catégorie d’image est un problème de classification ; prévoir une consommation ou une durée est un problème de régression.


![Comparaison classification régression et non supervisé](../../ressources/illustrations/jour_01/01_types_problemes.png)

*Le type de réponse attendue détermine la famille du problème avant le choix du modèle.*


## Le type de problème vient de la réponse attendue

On ne choisit pas d’abord un algorithme. On commence par définir ce que devra produire le système.

- « panne ou pas de panne » possède deux classes : c’est une **classification binaire** ;
- « type de panne A, B ou C » possède plusieurs classes : c’est une **classification multiclasse** ;
- « température dans une heure » est une valeur mesurable : c’est une **régression** ;
- « quels groupes de machines se ressemblent ? » n’impose pas de bonne réponse connue : c’est un problème **non supervisé**.

En apprentissage supervisé, chaque ligne d’entraînement doit être associée à une cible fiable. Un label faux ou ambigu enseigne au modèle une mauvaise réponse, même si le code est correct.


### Préparer les outils, puis ouvrir le tableau

`import` rend disponible une bibliothèque, c'est-à-dire un ensemble d'outils déjà écrits. `as pd`, par exemple, lui donne un nom court. **pandas** lit et manipule les tableaux ; **NumPy** calcule avec des tableaux de nombres ; **Matplotlib** dessine. Une **variable** est un nom associé à une valeur : `df` désignera notre tableau.

Exécutez les cellules de haut en bas, avec **Maj + Entrée**. Une cellule terminée peut produire un texte, un tableau ou un graphique ; l'absence de sortie n'est pas une erreur.


In [ ]:
from pathlib import Path  # Path décrit un chemin de fichier, sur Windows comme sur Linux.
import numpy as np  # np désigne les outils de calcul numérique.
import pandas as pd  # pd désigne les outils de tableaux.
import matplotlib.pyplot as plt  # plt désigne les outils de dessin.
from IPython.display import display  # display affiche joliment un tableau dans Jupyter.

plt.style.use("seaborn-v0_8-whitegrid")  # Ajoute une grille lisible aux graphiques.


Le petit outil suivant retrouve le dossier du cours. `def` définit une **fonction**, un bloc réutilisable ; `for` essaie plusieurs dossiers ; `if` teste une condition ; `return` fournit le résultat. `Path.cwd()` est le dossier courant et `.parents` ses dossiers parents. Le `/` entre chemins assemble les noms, il ne fait pas une division.

C'est une étape technique de chargement, pas une méthode de Machine Learning. Vous pouvez exécuter cette fonction telle quelle.


In [ ]:
def find_course_root():
    """Chercher le dossier qui contient les données du cours."""
    current = Path.cwd()  # Dossier depuis lequel Jupyter exécute ce notebook.
    for candidate in (current, *current.parents):  # * ajoute les parents à la liste d'essais.
        data_path = candidate / "ressources" / "donnees" / "maintenance_machines.csv"
        if data_path.exists():  # Arrêter la recherche dès que le fichier est trouvé.
            return candidate
    raise FileNotFoundError("Ouvrez le notebook depuis le dossier fondamentaux_ml_dl.")

ROOT = find_course_root()  # ROOT garde le chemin du dossier contenant les données.
DATA_FILE = ROOT / "ressources" / "donnees" / "maintenance_machines.csv"


In [ ]:
df = pd.read_csv(DATA_FILE)  # Lire le CSV : df est maintenant un DataFrame (tableau).
display(df.head(3))  # Montrer 3 lignes seulement ; le tableau complet n'est pas modifié.


## Lire le profil des colonnes

`dtypes` indique le type stocké dans chaque colonne : entier, nombre décimal ou texte. `isna().sum()` compte les valeurs manquantes ; une valeur manquante est une information absente, pas un zéro.

`describe()` calcule quelques repères sur les colonnes numériques : nombre de valeurs, moyenne, écart-type, minimum, quartiles et maximum. Le `.T` transpose simplement le tableau pour placer une variable par ligne et faciliter la lecture.

Ces statistiques servent à repérer une unité inattendue, une colonne constante ou une valeur très éloignée. Elles ne décident pas automatiquement qu’une valeur est fausse : il faut la confronter au sens métier.


**Pour lire les statistiques :** une moyenne est un niveau moyen ; l'**écart-type** mesure la dispersion autour de ce niveau. Les **quartiles** partagent les valeurs triées en quatre groupes : `50%` est la médiane, la valeur qui coupe les données en deux. Ici, il n'y a pas de valeur manquante ; ce n'est pas garanti avec de vrais fichiers.


In [ ]:
display(df.dtypes)  # Voir le format de chaque colonne : texte, entier, décimal...
missing = df.isna().sum()  # isna repère les absences ; sum les compte par colonne.
print("Valeurs manquantes :", int(missing.sum()))  # Additionner les comptes des colonnes.
display(df.describe().T)  # Résumer les nombres ; T place une variable par ligne.


## Construire la matrice X et le vecteur y

`X` est un tableau avec une ligne par observation et une colonne par feature. `y` contient une seule réponse par ligne. L'identifiant de machine n'est pas une mesure physique utile ici : le donner au modèle pourrait l'inciter à mémoriser des identifiants au lieu d'apprendre des relations générales.

La cible ne doit jamais être incluse dans `X`. Sinon, le modèle recevrait directement la réponse : ce serait une **fuite de cible**.


### Retrouver les entrées et la réponse

Une liste Python utilise des crochets : `["a", "b"]`. `df[features]` sélectionne plusieurs colonnes et garde un tableau 2D ; `df["failure_7d"]` sélectionne une colonne et produit une **Series**, une suite 1D de réponses. Les lignes de `X` et `y` doivent décrire exactement les mêmes observations, dans le même ordre.


In [ ]:
features = [  # Noms des mesures disponibles à l'instant de la décision.
    "temperature_c", "vibration_mm_s", "pressure_bar",
    "load_pct", "age_years", "days_since_maintenance",
]
X = df[features]  # Entrées : une ligne par observation, 6 colonnes par ligne.
y = df["failure_7d"]  # Réponses connues : 0 = pas de panne ; 1 = panne dans 7 jours.
print("X :", X.shape, "| y :", y.shape)  # shape donne les dimensions, pas les valeurs.


### Vérifier ce que signifie la forme

![Sélection de colonnes et correspondance des lignes](../../ressources/illustrations/jour_01/01_indexation_concrete.png)

Si l'on choisit 6 mesures pour 3 observations, `X.shape` vaut `(3, 6)` et `y.shape` vaut `(3,)`. La virgule de `(3,)` signale une forme avec une seule dimension. Dans une colonne de 0 et de 1, `y.mean()` donne le taux de 1 : par exemple `(0 + 1 + 0 + 1) / 4 = 50 %`.

**Avant d'exécuter :** si on retire une feature, quelle dimension change ? Les réponses et le nombre de lignes doivent-ils changer ?


In [ ]:
print("Taux de panne :", f"{y.mean():.1%}")  # .1% affiche un pourcentage à une décimale.
print("Cible accidentellement incluse dans X ?", "failure_7d" in X.columns)  # Doit être False.


## Une vraie démonstration de régression

La classification répond par une **classe**. La régression répond par un **nombre continu**. Nous allons demander à une régression linéaire d'estimer la température d'une machine à partir de sa charge et de son âge.

Une **régression linéaire** cherche une relation simple : elle combine les variables d'entrée avec des coefficients appris. Elle sert ici de premier repère, pas de vérité universelle.


![Droite de régression et résidus](../../ressources/illustrations/jour_01/01_regression_residus.png)

Une régression linéaire calcule une prédiction en additionnant une valeur de départ, appelée **interception** ou biais, et une contribution pour chaque feature. Chaque contribution dépend d’un **coefficient appris**.

On peut lire l’idée ainsi : `prédiction = base + coefficient × charge + coefficient × âge`. Les coefficients ne sont pas choisis à la main ; `fit` les ajuste sur les exemples d’entraînement. `predict` applique ensuite la relation apprise à de nouvelles lignes.

La ligne orange du schéma représente la relation moyenne apprise. Les points ne tombent pas tous exactement dessus, car deux features ne suffisent pas à expliquer toute la température. La distance entre un point réel et sa prédiction est le **résidu**.


### Pourquoi séparer aussi cette mini-régression ?

Le modèle apprend sur `X_reg_train` et `y_reg_train`. Les 25 % placés dans `X_reg_test` et `y_reg_test` restent invisibles pendant `fit` et servent à vérifier les prédictions.

`test_size=0.25` fixe cette proportion et `random_state=42` rend le tirage reproductible. Le notebook suivant expliquera en détail pourquoi un projet complet ajoute aussi un ensemble de validation.


### 1. Changer de question, donc changer de cible

Dans cet exemple séparé, nous prédisons la **température**, un nombre. La charge et l'âge deviennent les entrées. Les doubles crochets `df[[...]]` gardent un tableau de deux colonnes ; un seul nom sélectionne la cible. **scikit-learn**, importé sous `sklearn`, fournit les modèles et leur évaluation.


In [ ]:
from sklearn.model_selection import train_test_split  # Séparer les exemples.
from sklearn.linear_model import LinearRegression  # Modèle qui combine linéairement les entrées.
from sklearn.metrics import mean_absolute_error, mean_squared_error  # Mesurer les écarts.

X_regression = df[["load_pct", "age_years"]]  # Deux informations pour estimer la température.
y_regression = df["temperature_c"]  # Nouvelle cible numérique ; distincte de la panne.
X_reg_train, X_reg_test, y_reg_train, y_reg_test = train_test_split(
    X_regression, y_regression, test_size=0.25, random_state=42
)


### 2. Créer, apprendre, puis prédire

`LinearRegression()` crée un modèle **non entraîné**. `fit(X, y)` apprend les coefficients à partir des entrées et des réponses associées. `predict(X)` applique ensuite ces coefficients ; on ne lui donne pas les bonnes réponses du test.


In [ ]:
regression = LinearRegression()  # Créer l'objet qui contiendra les coefficients appris.
regression.fit(X_reg_train, y_reg_train)  # Apprendre uniquement avec les lignes du train.
temperature_predite = regression.predict(X_reg_test)  # Produire une valeur par ligne inconnue.
print("Premières prédictions (°C) :", temperature_predite[:3])  # :3 garde les positions 0, 1 et 2.


### Comment lire le résultat ?

L'écart entre la valeur réelle et la valeur prédite s'appelle une **erreur**, ou un **résidu**.

- La **MAE** est l'erreur absolue moyenne. Une MAE de 2 °C signifie qu'en moyenne les prédictions s'écartent d'environ 2 °C.
- La **RMSE** pénalise davantage les grosses erreurs. Si elle est nettement supérieure à la MAE, quelques prédictions sont probablement très mauvaises.
- Sur le graphique, plus un point est proche de la diagonale, plus la prédiction est juste.

Une métrique ne suffit jamais : il faut aussi regarder les erreurs concrètes et vérifier si elles sont acceptables pour l'usage prévu.


### MAE et RMSE avec un exemple simple

Imaginons trois erreurs absolues de `1`, `2` et `6` °C. La MAE les moyenne directement : la grosse erreur compte pour 6. La RMSE met d’abord les erreurs au carré : l’erreur de 6 °C influence donc beaucoup plus le résultat.

La MAE est souvent facile à expliquer dans l’unité métier. La RMSE attire davantage l’attention sur les grandes erreurs. Aucune des deux ne dit à elle seule pourquoi le modèle se trompe.

Enfin, une relation statistique n’est pas nécessairement une cause. Si charge et température évoluent ensemble, cela ne prouve pas que la charge soit la seule cause de la température. Le modèle décrit les données disponibles ; l’expertise métier reste nécessaire pour interpréter.


### 3. Calculer les deux erreurs annoncées

Une **erreur absolue** ignore le signe : prédire 48 °C pour 50 °C fait une erreur de 2 °C. `mean_absolute_error` moyenne ces distances. `mean_squared_error` moyenne les erreurs **au carré** ; `np.sqrt` prend la racine carrée pour revenir en °C.

Dans l'exemple `1, 2, 6`, la MAE vaut `9 / 3 = 3 °C` ; la RMSE vaut `racine((1 + 4 + 36) / 3) ≈ 3,70 °C`. Nous mesurons ici le test d'une démonstration dont les choix sont déjà fixés : nous ne l'utilisons pas pour régler le modèle.


In [ ]:
mae = mean_absolute_error(y_reg_test, temperature_predite)  # Écart moyen en °C.
rmse = np.sqrt(mean_squared_error(y_reg_test, temperature_predite))  # Accent sur les gros écarts.
print("MAE :", round(mae, 2), "°C")  # round limite seulement le nombre de décimales affichées.
print("RMSE :", round(rmse, 2), "°C")


### 4. Voir les erreurs, pas seulement un score

Chaque point compare une température **réelle** (axe horizontal) et **prédite** (axe vertical). Sur la diagonale, les deux valeurs sont égales. `scatter` dessine les points, `plot` la ligne de référence. `alpha=0.45` rend les points transparents pour voir ceux qui se superposent.


In [ ]:
plt.figure(figsize=(6, 5))  # Créer une figure de 6 pouces par 5.
plt.scatter(y_reg_test, temperature_predite, alpha=0.45)  # Une paire réel/prédit par point.
bornes = [  # Étendre la diagonale du minimum au maximum observé.
    min(y_reg_test.min(), temperature_predite.min()),
    max(y_reg_test.max(), temperature_predite.max()),
]
plt.plot(bornes, bornes, "--", color="black", label="prédiction parfaite")
plt.xlabel("Température réelle (°C)")  # Nommer les axes et leurs unités.
plt.ylabel("Température prédite (°C)")
plt.title("Régression : valeurs réelles et prédites")
plt.legend()  # Associer la ligne pointillée à sa légende.
plt.show()  # Afficher la figure.


## Le déséquilibre des classes

Les pannes sont heureusement moins fréquentes que les situations normales. Un modèle qui répond toujours « pas de panne » peut donc obtenir une exactitude élevée tout en étant inutile. Cette observation justifiera l'usage de plusieurs métriques et d'une baseline.


## À vous de jouer — sélectionner sans fuite

1. Avant le code : vibration, température et jours depuis maintenance sont-ils connus **avant** la panne ? La cible doit-elle figurer parmi ces trois colonnes ?
2. Complétez la liste `colonnes_exercice` avec leurs noms exacts, puis `cible_exercice` avec le nom de la réponse.
3. La cellule affiche les formes une fois les noms renseignés. Attendez-vous à **3 colonnes dans X**, et toujours une réponse par ligne.


In [ ]:
# Solution guidée
colonnes_exercice = ["vibration_mm_s", "temperature_c", "days_since_maintenance"]
cible_exercice = "failure_7d"
X_exercice = df[colonnes_exercice]  # Un tableau de trois mesures par ligne.
y_exercice = df[cible_exercice]  # Une suite de réponses dans le même ordre.
print("Formes :", X_exercice.shape, y_exercice.shape)
assert X_exercice.shape == (len(df), 3)  # Vérifier le nombre de mesures retenues.
assert cible_exercice not in X_exercice.columns  # Vérifier que la réponse n'est pas une entrée.


**Pourquoi cette réponse ?** Les trois mesures peuvent décrire l'état actuel. `failure_7d` est une réponse constatée après cet état : elle reste dans y. Sélectionner les colonnes ne retire ni ne réordonne les lignes ; X et y restent alignés.


**À retenir :** le modèle ne doit recevoir que des informations réellement disponibles au moment où l'on souhaite prédire.
